In [1]:
import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

RAW_DATA = PROJECT_ROOT / "FIFA WC Dataset"
CLEANED_DATA = PROJECT_ROOT / "cleaned_data"
PROCESSED_DATA = PROJECT_ROOT / "processed_data"
FEATURE_DATA = PROJECT_ROOT / "feature_data"

results = pd.read_csv(CLEANED_DATA / "results_2000_fifa211.csv")
team_matches = pd.read_csv(PROCESSED_DATA / "team_matches.csv")
saved_stats = pd.read_csv(PROCESSED_DATA / "team_stats.csv")
elo = pd.read_csv(PROCESSED_DATA / "elo_ratings_V3.csv")

In [2]:
results["date"] = pd.to_datetime(results["date"], dayfirst = True)
results = results.sort_values("date").reset_index(drop = True)
results.head()

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
0,2000-01-04,Egypt,Togo,2,1,Friendly,Aswan,Egypt,False
1,2000-01-07,Tunisia,Togo,7,0,Friendly,Tunis,Tunisia,False
2,2000-01-08,Trinidad and Tobago,Canada,0,0,Friendly,Port of Spain,Trinidad and Tobago,False
3,2000-01-09,Burkina Faso,Gabon,1,1,Friendly,Ouagadougou,Burkina Faso,False
4,2000-01-09,Guatemala,Armenia,1,1,Friendly,Los Angeles,United States,True


In [3]:
match_features = []
team_state = {}

In [4]:
elo.head()

,rank,team,elo_rating
0,1,Argentina,2075.73
1,2,France,2060.12
2,3,Spain,2035.55
3,4,England,1998.84
4,5,Morocco,1948.02


In [5]:
for team in elo["team"]:
    
    team_state[team] = {

        "matches": 0,
        "wins": 0,
        "draws": 0,
        "losses": 0,

        "goals_for": 0,
        "goals_against": 0,

        "elo": 1500
    }

In [6]:
team_state["Argentina"]

{'matches': 0,
 'wins': 0,
 'draws': 0,
 'losses': 0,
 'goals_for': 0,
 'goals_against': 0,
 'elo': 1500}

In [7]:
# ----- Elo V3 helper Functions -----

HOME_ADVANTAGE = 100

def get_k_value(tournament):

    if tournament == "Friendly":
        return 10

    elif tournament == "FIFA World Cup":
        return 60

    elif tournament == "FIFA World Cup qualification":
        return 40

    elif tournament in [
        "UEFA Euro",
        "Copa América",
        "AFC Asian Cup",
        "African Cup of Nations",
        "CONCACAF Gold Cup",
        "OFC Nations Cup"
    ]:
        return 35

    elif "qualification" in tournament.lower():
        return 25

    elif "nations league" in tournament.lower():
        return 25

    else:
        return 20
    

def goal_difference_multiplier(goal_difference):
    """
    Returns the Elo multiplier based on goal difference.
    """

    if goal_difference == 1:
        return 1.00
    elif goal_difference == 2:
        return 1.50
    elif goal_difference == 3:
        return 1.75
    elif goal_difference == 4:
        return 2.00
    else:
        return 2.25

In [8]:
for _, match in results.iterrows():

    home_team = match["home_team"]
    away_team = match["away_team"]

    home_score = match["home_score"]
    away_score = match["away_score"]

    home = team_state[home_team]
    away = team_state[away_team]

    home_elo = home["elo"]
    away_elo = away["elo"]
    elo_diff = home_elo - away_elo

# Home & Away Matches played
    home_matches_played = home["matches"]
    away_matches_played = away["matches"]

# Home win percentage
    if home["matches"] == 0:
        home_win_pct = 0
    else:
        home_win_pct = home["wins"] / home["matches"]

# Away win percentage
    if away["matches"] == 0:
        away_win_pct = 0
    else:
        away_win_pct = away["wins"] / away["matches"]

# Home average goals scored
    if home["matches"] == 0:
        home_avg_gf = 0
    else:
        home_avg_gf = home["goals_for"] / home["matches"]

# Away average goals scored
    if away["matches"] == 0:
        away_avg_gf = 0
    else:
        away_avg_gf = away["goals_for"] / away["matches"]

# Home average goals conceded
    if home["matches"] == 0:
        home_avg_ga = 0
    else:
        home_avg_ga = home["goals_against"] / home["matches"]

# Away average goals conceded
    if away["matches"] == 0:
        away_avg_ga = 0
    else:
        away_avg_ga = away["goals_against"] / away["matches"]

# Match information
    date = match["date"]
    neutral = int(match["neutral"])
    tournament = match["tournament"]
        
# Target
    if home_score > away_score:
        target = 2

    elif home_score < away_score:
        target = 0

    else:
        target = 1

    feature_row = {
    "date": date,
    "home_team": home_team,
    "away_team": away_team,
    
    "home_score": home_score,
    "away_score": away_score,
    "goal_difference": home_score - away_score,
    
    "home_elo": home_elo,
    "away_elo": away_elo,
    "elo_diff": elo_diff,

    "home_matches_played": home_matches_played,
    "away_matches_played": away_matches_played,

    "home_win_pct": home_win_pct,
    "away_win_pct": away_win_pct,

    "home_avg_gf": home_avg_gf,
    "away_avg_gf": away_avg_gf,

    "home_avg_ga": home_avg_ga,
    "away_avg_ga": away_avg_ga,

    "neutral": neutral,
    "tournament": tournament,

    "target": target
}
    
    match_features.append(feature_row)
    
    
# Update matches played
    home["matches"] += 1
    away["matches"] += 1


# Update wins, draws and losses
    if home_score > away_score:
        home["wins"] += 1
        away["losses"] += 1

    elif home_score < away_score:
        away["wins"] += 1
        home["losses"] += 1

    else:
        home["draws"] += 1
        away["draws"] += 1

# Update goals scored and conceded
    home["goals_for"] += home_score
    home["goals_against"] += away_score

    away["goals_for"] += away_score
    away["goals_against"] += home_score
    #print(home_team, away_team)

# ------------------------------------
# Update Elo Rating (V3)
# ------------------------------------
# Current ratings

    home_rating = home["elo"]
    away_rating = away["elo"]

# Tournament K-factor
    k = get_k_value(tournament)
    
    
# Goal difference multiplier
    goal_difference = abs(home_score - away_score)
    gd_multiplier = goal_difference_multiplier(goal_difference)


# Apply home advantage only for non-neutral matches
    effective_home_rating = home_rating
    effective_away_rating = away_rating

    if not neutral:
        effective_home_rating += HOME_ADVANTAGE


# Expected scores
    expected_home = 1 / (
        1 + 10 ** (
            (effective_away_rating - effective_home_rating) / 400
        )
    )

    expected_away = 1 - expected_home
    
# Actual scores
    if home_score > away_score:
        actual_home = 1
        actual_away = 0

    elif home_score < away_score:
        actual_home = 0
        actual_away = 1

    else:
        actual_home = 0.5
        actual_away = 0.5

# New ratings
    new_home_rating = (
        home_rating +
        k * gd_multiplier * (actual_home - expected_home)
    )

    new_away_rating = (
        away_rating +
        k * gd_multiplier * (actual_away - expected_away)
    )

    # Store updated Elo ratings
    home["elo"] = new_home_rating
    away["elo"] = new_away_rating
    

In [9]:
match_features_df = pd.DataFrame(match_features)

match_features_df.shape
match_features_df.head()
match_features_df.info()
match_features_df.isna().sum()

<class 'pandas.DataFrame'>
RangeIndex: 23760 entries, 0 to 23759
Data columns (total 20 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   date                 23760 non-null  datetime64[us]
 1   home_team            23760 non-null  str           
 2   away_team            23760 non-null  str           
 3   home_score           23760 non-null  int64         
 4   away_score           23760 non-null  int64         
 5   goal_difference      23760 non-null  int64         
 6   home_elo             23760 non-null  float64       
 7   away_elo             23760 non-null  float64       
 8   elo_diff             23760 non-null  float64       
 9   home_matches_played  23760 non-null  int64         
 10  away_matches_played  23760 non-null  int64         
 11  home_win_pct         23760 non-null  float64       
 12  away_win_pct         23760 non-null  float64       
 13  home_avg_gf          23760 non-null  float

date                   0
home_team              0
away_team              0
home_score             0
away_score             0
goal_difference        0
home_elo               0
away_elo               0
elo_diff               0
home_matches_played    0
away_matches_played    0
home_win_pct           0
away_win_pct           0
home_avg_gf            0
away_avg_gf            0
home_avg_ga            0
away_avg_ga            0
neutral                0
tournament             0
target                 0
dtype: int64

In [10]:
match_features_df.tail()

,date,home_team,away_team,home_score,away_score,goal_difference,home_elo,away_elo,elo_diff,home_matches_played,away_matches_played,home_win_pct,away_win_pct,home_avg_gf,away_avg_gf,home_avg_ga,away_avg_ga,neutral,tournament,target
23755,2026-07-05,Mexico,England,2,3,-1,1914.033707,1964.601665,-50.567958,473,321,0.543340,0.601246,1.672304,1.984424,0.947146,0.775701,0,FIFA World Cup,0
23756,2026-07-06,Portugal,Spain,0,1,-1,1940.716327,2011.585230,-70.868903,336,345,0.595238,0.686957,2.089286,2.200000,0.833333,0.707246,1,FIFA World Cup,0
23757,2026-07-06,USA,Belgium,1,4,-3,1812.318122,1861.830427,-49.512304,423,290,0.541371,0.551724,1.749409,1.993103,0.990544,1.044828,0,FIFA World Cup,0
23758,2026-07-07,Argentina,Egypt,3,2,1,2064.934264,1801.439090,263.495173,342,357,0.631579,0.571429,1.932749,1.714286,0.824561,0.837535,1,FIFA World Cup,2
23759,2026-07-07,Switzerland,Colombia,0,0,0,1882.616182,1957.738735,-75.122554,290,326,0.465517,0.481595,1.665517,1.374233,1.075862,0.868098,1,FIFA World Cup,1


In [11]:
match_features_df.shape

(23760, 20)

In [12]:
match_features_df.loc[
    match_features_df["home_team"] == "Egypt",
    ["date", "home_elo", "home_matches_played", "home_win_pct"]
].head(10)

,date,home_elo,home_matches_played,home_win_pct
0,2000-01-04,1500.000000,0,0.000000
25,2000-01-23,1494.399288,3,0.333333
41,2000-01-28,1521.072405,4,0.500000
57,2000-02-01,1538.681094,5,0.600000
87,2000-02-07,1559.912270,6,0.666667
310,2000-04-09,1539.698378,7,0.571429
368,2000-04-23,1558.246451,8,0.625000
530,2000-06-09,1577.771603,10,0.600000
566,2000-06-17,1572.540070,11,0.545455
753,2000-08-25,1581.698221,13,0.538462


In [13]:
match_features_df.loc[
    match_features_df["away_team"] == "Egypt",
    ["date", "away_elo", "away_matches_played", "away_win_pct"]
].head(10)

,date,away_elo,away_matches_played,away_win_pct
5,2000-01-09,1503.599350,1,1.000000
16,2000-01-19,1498.128518,2,0.500000
525,2000-06-07,1574.174860,9,0.666667
652,2000-07-09,1576.583701,12,0.583333
829,2000-10-02,1592.248845,15,0.600000
871,2000-10-08,1597.909860,16,0.625000
1056,2001-01-17,1626.977572,20,0.700000
1154,2001-02-24,1605.299493,23,0.652174
1234,2001-03-23,1613.775492,26,0.615385
1586,2001-06-10,1609.821854,31,0.612903


In [14]:
match_features_df.to_csv(FEATURE_DATA / "match_feature_V1.csv", index = False)

## Match Features V1 (Completed)

Created a chronological match-level feature dataset (`match_features_v1.csv`) for machine learning. Each row represents a single match and contains only information available **before kickoff**, preventing data leakage.

### Features Included
- Pre-match Elo ratings (V3)
- Elo difference
- Matches played
- Win percentage
- Average goals scored
- Average goals conceded
- Neutral venue flag
- Tournament
- Target (Home Win = 2, Draw = 1, Away Win = 0)

### Validation
- 25,386 matches processed successfully
- No missing values
- Chronological feature generation verified
- Elo updates matched V3 implementation
- Saved with `index=False`

also want to remind the reader, World Cup 2026 matches are still being played. all these datasets will be updated daily.